# Cohort Analysis with RLM + DataFrame

This notebook demonstrates how to pass large DataFrames to a DSPy RLM (Recursive Language Model) for iterative data analysis.

We'll give the RLM three DataFrames from a SaaS product — users, events, and subscriptions — and ask it to investigate why retention is dropping. The RLM will iteratively explore the data, writing and executing Python code in a sandbox, until it identifies the root cause.

**Expected findings:**
- Overall churn rate: ~23-25%
- Worst channel: `paid_campaign_x` (~45% churn vs ~15% organic)
- Key insight: churned users from `paid_campaign_x` never use `advanced_reports` — early usage of that feature is an activation signal

**Prerequisites:**
1. Run `generate_cohort_data.py` to create the parquet files
2. Set your LLM API key (e.g. `OPENAI_API_KEY` or `ANTHROPIC_API_KEY`)

## 1. Generate the dataset

Run `generate_cohort_data.py` to create three parquet files with embedded bias signals. You only need to do this once.

In [1]:
!python generate_cohort_data.py

Generating users...
Generating subscriptions...
Generating events...

Dataset stats:
  Users:             10,000 rows
  Subscriptions:     10,000 rows
  Events:           250,918 rows
  Total memory:        16.3 MB

Embedded signal:
  paid_campaign_x churn rate: 45%
  organic churn rate:         16%

Saved to: users.parquet, subscriptions.parquet, events.parquet


## 2. Imports and setup

In [2]:
import pandas as pd

import dspy

# Import the standalone DataFrame wrapper (implements SandboxSerializable)
from dataframe import DataFrame

## 3. Load and inspect the data

In [3]:
users = pd.read_parquet("users.parquet")
events = pd.read_parquet("events.parquet")
subscriptions = pd.read_parquet("subscriptions.parquet")

print(f"Users:         {len(users):,} rows")
print(f"Events:        {len(events):,} rows")
print(f"Subscriptions: {len(subscriptions):,} rows")

Users:         10,000 rows
Events:        250,918 rows
Subscriptions: 10,000 rows


In [4]:
users.head()

,user_id,email,name,signup_date,acquisition_channel,plan_at_signup,country
0,1,johnsonjoshua@example.org,Brian Yang,2024-01-29,organic,free,US
1,2,garzaanthony@example.org,Jonathan Johnson,2024-01-27,paid_campaign_x,pro,US
2,3,jennifermiles@example.com,Kevin Pacheco,2024-04-18,organic,free,US
3,4,blakeerik@example.com,Christopher Bernard,2024-01-07,paid_campaign_x,pro,AU
4,5,curtis61@example.com,Lindsey Roman,2024-04-17,organic,starter,BR


In [5]:
subscriptions.head()

,user_id,subscription_start,subscription_end,plan,mrr,status,cancellation_reason
0,1,2024-01-29,NaT,free,0,active,NaN
1,2,2024-01-27,NaT,pro,79,active,NaN
2,3,2024-04-18,NaT,free,0,active,NaN
3,4,2024-01-07,NaT,pro,79,active,NaN
4,5,2024-04-17,2024-05-09,starter,29,cancelled,poor_support


## 4. Wrap DataFrames for RLM sandbox injection

The `DataFrame` wrapper implements `SandboxSerializable`, which tells the RLM how to:
- **Serialize** the data (base64-encoded Parquet)
- **Reconstruct** it inside the sandbox (`pd.read_parquet(...)`)
- **Preview** it for the LLM (column names, dtypes, sample rows)

The full data is injected once into the sandbox at the start of the session. The LLM sees only a compact preview in its prompt but can write code against the real DataFrame.

In [6]:
wrapped_users = DataFrame(users)
wrapped_events = DataFrame(events)
wrapped_subscriptions = DataFrame(subscriptions)

# This is what the LLM sees in its prompt:
print(wrapped_users.rlm_preview())

DataFrame: 10,000 rows x 7 columns

Columns:
  user_id: int64
  email: str
  name: str
  signup_date: datetime64[us]
  acquisition_channel: str
  plan_at_signup: str
  country: str

Sample (first 3 rows):
   user_id                      email              name signup_date acquisition_channel plan_at_signup country
0        1  johnsonjoshua@example.org        Brian Yang  2024-01-29             organic           free      US
1        2   garzaanthony@example.org  Jonathan Johnson  2024-01-27     p...


## 5. Define the signature

The signature declares three `DataFrame` inputs and four structured outputs. The docstring becomes the system prompt for the RLM.

In [7]:
class CohortRetentionAnalysis(dspy.Signature):
    """You are a data analyst investigating why user retention is dropping.

    You have access to three DataFrames:
    - `users`: user profiles with signup dates and acquisition channels
    - `events`: feature usage events with timestamps
    - `subscriptions`: subscription status, plan, MRR, and cancellation reasons

    Investigate the data step by step. Compute retention by cohort, segment by
    acquisition channel, compare feature usage between retained and churned users,
    and identify the root cause of churn.
    """

    users: DataFrame = dspy.InputField(desc="User profiles with signup_date, acquisition_channel, country")
    events: DataFrame = dspy.InputField(desc="Feature usage events with user_id, event_type, timestamp")
    subscriptions: DataFrame = dspy.InputField(desc="Subscription records with status, plan, mrr, cancellation_reason")

    overall_churn_rate: float = dspy.OutputField(desc="Overall churn rate as a decimal (e.g. 0.25 for 25%)")
    worst_channel: str = dspy.OutputField(desc="Acquisition channel with highest churn rate")
    key_finding: str = dspy.OutputField(desc="The main insight about what differentiates churned users")
    recommendations: str = dspy.OutputField(desc="2-3 actionable recommendations based on the analysis")

## 6. Configure the LM and run

Update the model name below to match your provider. The RLM will run up to 15 iterations, writing and executing Python code in a sandboxed interpreter at each step.

In [8]:
lm = dspy.LM("openai/gpt-5.4")  # or "anthropic/claude-sonnet-4-5", etc.
dspy.configure(lm=lm)

In [9]:
analyzer = dspy.RLM(
    CohortRetentionAnalysis,
    max_iterations=15,
    verbose=True,
)

result = analyzer(
    users=wrapped_users,
    events=wrapped_events,
    subscriptions=wrapped_subscriptions,
)

/Users/kmad/dev/dspy-fresh/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st... None}, annotations=[])), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
2026/03/08 15:52:10 INFO dspy.predict.rlm: RLM iteration 1/15
Reasoning: First step is to explore the data structure and key distributions: shapes, date ranges, status values, acquisition channels, plans, nulls, and some sample cancellation reasons/event types. Then I can define churn/retention properly and compute cohort analyses in later steps.
Code:
```python
p

## 7. Results

In [10]:
print(f"Overall Churn Rate: {result.overall_churn_rate:.1%}")
print(f"Worst Channel:     {result.worst_channel}")
print(f"\nKey Finding:\n  {result.key_finding}")
print(f"\nRecommendations:\n  {result.recommendations}")
print(f"\nRLM completed in {len(result.trajectory)} iterations")

Overall Churn Rate: 23.8%
Worst Channel:     paid_campaign_x

Key Finding:
  Churn is 23.77% overall, and paid_campaign_x is the clear outlier with 44.76% churn across every signup cohort and plan. The strongest behavioral signal is early adoption of advanced_reports: users who adopt it churn far less overall (14.7% vs 27.7%), while paid_campaign_x has by far the lowest advanced_reports adoption (18.7% vs ~33-35% for other channels). Within paid_campaign_x, churn is concentrated among users who never adopt advanced_reports, indicating the retention drop is primarily an acquisition-quality/onboarding problem rather than a broad product usage decline.

Recommendations:
  Pause or tighten paid_campaign_x targeting; redesign onboarding for campaign_x users to drive advanced_reports activation in the first 14 days; align ad and landing-page messaging to the product's core reporting use case; and track weekly activation-to-retention funnels by channel with advanced_reports adoption as the le